#Generalisation Testing

In [1]:
!pip install -q kagglehub

import kagglehub

path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(path)

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.
/kaggle/input/140k-real-and-fake-faces


##Download and Setup

In [2]:
import os

DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"

print(DATASET_PATH)
print(os.listdir(DATASET_PATH))

/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
['fake', 'real']


##Data Preparation

Prepares the same balanced StyleGAN-based dataset

In [4]:
import os
import random
import torch

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder



DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"


train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])


full_dataset = ImageFolder(DATASET_PATH, transform=None)

print("Classes:", full_dataset.classes)
print("Class mapping:", full_dataset.class_to_idx)
print("Total available:", len(full_dataset))


random.seed(42)
torch.manual_seed(42)  #reproducibility

real_label = full_dataset.class_to_idx["real"]
fake_label = full_dataset.class_to_idx["fake"]

real_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == real_label
]

fake_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == fake_label
]

random.shuffle(real_indices)
random.shuffle(fake_indices)

selected_indices = real_indices[:8000] + fake_indices[:8000]
random.shuffle(selected_indices)

full_dataset.samples = [
    full_dataset.samples[i]
    for i in selected_indices
]

full_dataset.imgs = full_dataset.samples
full_dataset.targets = [
    label for _, label in full_dataset.samples
]

print("\nSelected dataset:")
print("Total:", len(full_dataset))
print(
    "Real:",
    sum(label == real_label for _, label in full_dataset.samples)
)
print(
    "Fake:",
    sum(label == fake_label for _, label in full_dataset.samples)
)


train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_sub, val_sub, test_sub = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)



class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[
            self.subset.indices[idx]
        ]

        image = self.subset.dataset.loader(path)

        return self.transform(image), label



train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)


train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(
    f"\nTrain: {len(train_set)}, "
    f"Val: {len(val_set)}, "
    f"Test: {len(test_set)}"
)

Classes: ['fake', 'real']
Class mapping: {'fake': 0, 'real': 1}
Total available: 100000

Selected dataset:
Total: 16000
Real: 8000
Fake: 8000

Train: 11200, Val: 2400, Test: 2400


In [8]:
import torch.nn as nn
import timm
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Mounted at /content/drive


##Defining Combined Architecture

In [9]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64*4*4, 128), nn.ReLU(inplace=True), nn.Dropout(0.3))
    def forward(self, x):
        fft = torch.fft.fft2(x)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2,-1), keepdim=True)) / (magnitude.std(dim=(-2,-1), keepdim=True) + 1e-6)
        phase = torch.angle(fft) / torch.pi
        freq_input = torch.cat([magnitude, phase], dim=1)
        out = self.features(freq_input)
        return self.classifier(out)

class CombinedDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.xception = timm.create_model('xception', pretrained=True, num_classes=0)
        xception_features = self.xception.num_features
        self.freq_branch = FrequencyBranch()
        self.classifier = nn.Sequential(
            nn.Linear(xception_features + 128, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, x_xception, x_freq):
        spatial_features = self.xception(x_xception)
        freq_features = self.freq_branch(x_freq)
        combined = torch.cat([spatial_features, freq_features], dim=1)
        return self.classifier(combined)

##Loading trained models

Both FaceForensics++ trained detectors, unmodified

In [10]:
combined_model = CombinedDetector().cuda()
combined_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_combinedwimpfftsplitaug.pth'))
combined_model.eval()
print("CombinedDetector loaded")

xception_model = timm.create_model('xception', pretrained=True, num_classes=2)
xception_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_baselinefix.pth'))
xception_model = xception_model.cuda()
xception_model.eval()
print("XceptionOnly loaded")

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
CombinedDetector loaded
XceptionOnly loaded


## Cross-Dataset Generalisation Test

Evaluates both models directly on the unseen
StyleGAN-based dataset, with no additional training.

*NOTE*  Naming error StyleGAN2, written instead of StyleGAN, no code or dataset error just naming issue

In [12]:
from sklearn.metrics import accuracy_score

def evaluate_cross_dataset(model, loader, dual_input=False):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images, images) if dual_input else model(images)
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    print(f"Cross-dataset accuracy (StyleGAN): {acc*100:.2f}%")
    return acc

print("CombinedDetector on StyleGAN (unseen during training):")
evaluate_cross_dataset(combined_model, test_loader, dual_input=True)

print("XceptionOnly on StyleGAN (unseen during training):")
evaluate_cross_dataset(xception_model, test_loader, dual_input=False)

CombinedDetector on StyleGAN2 (unseen during training):
Cross-dataset accuracy (StyleGAN2): 49.17%
XceptionOnly on StyleGAN2 (unseen during training):
Cross-dataset accuracy (StyleGAN2): 51.21%


0.5120833333333333